In [ ]:
# 01_free_particle.ipynb
# Simulation of a free particle in 1D
# using the split-operator algorithm implemented in Qiskit
# Reference: Kassal et al., PNAS 2008, arXiv:0801.2986

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Grid parameters
n_qubits = 4
N = 2**n_qubits
L = 20.0
dx = L / N
x = np.linspace(-L/2, L/2, N, endpoint=False)

In [ ]:
# Momentum grid (DFT order)
dp = 2 * np.pi / L
p = np.fft.fftfreq(N) * N * dp
#print(p)

In [ ]:
# Initial wavepacket parameters
x0 = -5.0
sigma = 1.0
p0 = 1.0

# Gaussian wavepacket
psi = np.exp(-(x - x0)**2 / (4 * sigma**2)) * np.exp(1j * p0 * x)

# Normalize
psi = psi / np.sqrt(np.sum(np.abs(psi)**2) * dx)

In [ ]:
# Plot initial wavepacket
plt.figure(figsize=(10, 4))
plt.plot(x, np.abs(psi)**2)
plt.xlabel("x")
plt.ylabel("|psi(x)|^2")
plt.title("Initial wavepacket")
plt.grid(True)
plt.show()

In [ ]:
# Time parameters
dt = 0.05
n_steps = 40

# Kinetic propagator phases
kinetic_phases = np.exp(-1j * p**2 / 2 * dt)

In [ ]:
# Time evolution with classical split-operator (FFT)
psi_t = psi.copy()
positions = []

for step in range(n_steps):
    # Transform to momentum space
    psi_p = np.fft.fft(psi_t)
    
    # Apply kinetic propagator
    psi_p = psi_p * kinetic_phases
    
    # Transform back to position space
    psi_t = np.fft.ifft(psi_p)
    
    # Store expectation value of position
    x_mean = np.sum(x * np.abs(psi_t)**2) * dx
    positions.append(x_mean)

In [ ]:
# Analytical solution: <x(t)> = x0 + p0 * t
times = np.arange(1, n_steps + 1) * dt
x_analytical = x0 + p0 * times

# Plot
plt.figure(figsize=(10, 4))
plt.plot(times, positions, label="Split-operator (FFT)")
plt.plot(times, x_analytical, "--", label="Analytical")
plt.xlabel("t")
plt.ylabel("<x>")
plt.title("Free particle: expectation value of position")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Check norm conservation
psi_t = psi.copy()
norms = []

for step in range(n_steps):
    psi_p = np.fft.fft(psi_t)
    psi_p = psi_p * kinetic_phases
    psi_t = np.fft.ifft(psi_p)
    norm = np.sum(np.abs(psi_t)**2) * dx
    norms.append(norm)

plt.figure(figsize=(10, 4))
plt.plot(times, norms)
plt.xlabel("t")
plt.ylabel("norm")
plt.title("Norm conservation")
plt.ylim(0.99, 1.01)
plt.grid(True)
plt.show()

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator
from qiskit.circuit.library import DiagonalGate
from qiskit.quantum_info import Statevector

In [ ]:
simulator = AerSimulator(method='statevector')

def qiskit_step(psi, kinetic_phases, n_qubits):
    
    qc = QuantumCircuit(n_qubits)
    qc.set_statevector(Statevector(psi))
    qc.append(QFTGate(n_qubits), range(n_qubits))
    qc.append(DiagonalGate(list(kinetic_phases)), range(n_qubits))
    qc.append(QFTGate(n_qubits).inverse(), range(n_qubits))
    qc.save_statevector()
    qc = transpile(qc, simulator)
    
    result = simulator.run(qc).result()
    psi_new = np.array(result.get_statevector())
    
    return psi_new

In [ ]:
psi_qiskit_norm = psi / np.sqrt(np.sum(np.abs(psi)**2))
psi_qiskit = qiskit_step(psi_qiskit_norm, kinetic_phases, n_qubits)
psi_fft = np.fft.ifft(np.fft.fft(psi_qiskit_norm) * kinetic_phases)
print("Max difference:", np.max(np.abs(psi_qiskit - psi_fft)))

In [ ]:
# Time evolution with Qiskit split-operator
psi_t = psi_qiskit_norm.copy()
positions_qiskit = []

for step in range(n_steps):
    psi_t = qiskit_step(psi_t, kinetic_phases, n_qubits)
    x_mean = np.sum(x * np.abs(psi_t)**2)
    positions_qiskit.append(x_mean)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(times, x_analytical, "--", label="Analytical")
plt.plot(times, positions, label="Classical FFT")
plt.plot(times, positions_qiskit, ".", label="Qiskit")
plt.xlabel("t")
plt.ylabel("<x>")
plt.title("Free particle: comparison classical vs Qiskit")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Time evolution storing full statevector
psi_t = psi_qiskit_norm.copy()
states = [psi_t.copy()]

for step in range(n_steps):
    psi_t = qiskit_step(psi_t, kinetic_phases, n_qubits)
    states.append(psi_t.copy())

In [ ]:
# Plot wavepacket evolution at different times
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)

steps_to_plot = [0, 10, 20, 40]

for ax, step in zip(axes, steps_to_plot):
    ax.plot(x, np.abs(states[step])**2)
    ax.set_title(f"t = {step * dt:.2f}")
    ax.set_xlabel("x")
    ax.grid(True)

axes[0].set_ylabel("|psi(x)|^2")
fig.suptitle("Free particle wavepacket evolution")
plt.tight_layout()
plt.show()